# 03 Models

**Project:** Interpretable vs Black-Box Models for Predicting Death Events in Heart Failure Patients

This notebook trains the models after preprocessing.  
Salem Part: Logistic Regression and Random Forest.  
Tala Part: Decision Tree and Gradient Boosting.

The proposal says to compare the models with and without the `time` feature, so I keep both versions here.


## Plan for this notebook

1. Load the preprocessed files from `results/tables/`.
2. Train Salem's Logistic Regression models.
3. Train Salem's Random Forest models.
4. Evaluate all Salem models using the same metrics.
5. Save Salem's results to `results/metrics/`.
6. Leave a clean section for Tala to add Decision Tree and Gradient Boosting.


In [ ]:
# Imports and setup

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    classification_report,
)

RANDOM_STATE = 42
TARGET_COL = "DEATH_EVENT"

# This makes the notebook work from either the project folder or the notebooks folder.
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

TABLES_DIR = PROJECT_ROOT / "results" / "tables"
METRICS_DIR = PROJECT_ROOT / "results" / "metrics"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"

METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Tables folder:", TABLES_DIR)
print("Metrics folder:", METRICS_DIR)
print("Figures folder:", FIGURES_DIR)


## Helper functions

I use the same evaluation function for every model so the comparison is fair.


In [ ]:
def load_table(file_name: str) -> pd.DataFrame:
    """Load a CSV file from results/tables."""
    file_path = TABLES_DIR / file_name

    if not file_path.exists():
        raise FileNotFoundError(f"Missing file: {file_path}")

    return pd.read_csv(file_path)


def load_y(file_name: str) -> pd.Series:
    """Load y_train or y_test safely, even if the CSV column name is different."""
    file_path = TABLES_DIR / file_name

    if not file_path.exists() or file_path.stat().st_size == 0:
        print(f"{file_name} is missing or empty. Trying to rebuild y from the raw dataset.")
        return rebuild_y_from_raw(file_name)

    y_df = pd.read_csv(file_path)

    if y_df.empty or y_df.shape[1] == 0:
        print(f"{file_name} did not load correctly. Trying to rebuild y from the raw dataset.")
        return rebuild_y_from_raw(file_name)

    if TARGET_COL in y_df.columns:
        y = y_df[TARGET_COL]
    else:
        # If the file has one unnamed column, this still works.
        y = y_df.iloc[:, -1]

    return y.astype(int).reset_index(drop=True)


def find_raw_dataset() -> Path:
    """Find the original heart failure CSV if y needs to be rebuilt."""
    possible_paths = [
        PROJECT_ROOT / "data" / "heart_failure_clinical_records_dataset.csv",
        PROJECT_ROOT / "data" / "heart_failure_clinical_records_dataset(4).csv",
        PROJECT_ROOT / "heart_failure_clinical_records_dataset.csv",
        PROJECT_ROOT / "heart_failure_clinical_records_dataset(4).csv",
        Path("/mnt/data/heart_failure_clinical_records_dataset(4).csv"),
        Path("/mnt/data/heart_failure_clinical_records_dataset.csv"),
    ]

    for path in possible_paths:
        if path.exists():
            return path

    raise FileNotFoundError("Could not find the raw heart failure dataset.")


def rebuild_y_from_raw(file_name: str) -> pd.Series:
    """Rebuild y_train or y_test using the saved train/test split indices."""
    raw_path = find_raw_dataset()
    df_raw = pd.read_csv(raw_path).drop_duplicates().reset_index(drop=True)

    if TARGET_COL not in df_raw.columns:
        raise ValueError(f"Could not find {TARGET_COL} in the raw dataset.")

    split_path = TABLES_DIR / "train_test_split_indices.csv"

    if not split_path.exists():
        raise FileNotFoundError(
            "y file is missing/empty and train_test_split_indices.csv was not found."
        )

    split_info = pd.read_csv(split_path)

    if "original_index" not in split_info.columns or "split" not in split_info.columns:
        raise ValueError(
            "train_test_split_indices.csv must have columns: original_index and split."
        )

    if file_name == "y_train.csv":
        needed_indices = split_info.loc[split_info["split"] == "train", "original_index"].to_numpy()
    elif file_name == "y_test.csv":
        needed_indices = split_info.loc[split_info["split"] == "test", "original_index"].to_numpy()
    else:
        raise ValueError("file_name must be y_train.csv or y_test.csv")

    y = df_raw.loc[needed_indices, TARGET_COL].astype(int).reset_index(drop=True)
    return y


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series, model_name: str, time_version: str) -> dict:
    """Evaluate one model and return the main metrics."""
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results = {
        "model": model_name,
        "time_version": time_version,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
    }

    return results


def save_confusion_matrix(model, X_test: pd.DataFrame, y_test: pd.Series, file_stem: str) -> None:
    """Save a confusion matrix plot."""
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Survived", "Death"])
    disp.plot()
    plt.title(file_stem.replace("_", " ").title())
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"confusion_matrix_{file_stem}.png", dpi=300)
    plt.show()


def save_roc_curve(model, X_test: pd.DataFrame, y_test: pd.Series, file_stem: str) -> None:
    """Save a ROC curve plot."""
    RocCurveDisplay.from_estimator(model, X_test, y_test)
    plt.title(file_stem.replace("_", " ").title())
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"roc_{file_stem}.png", dpi=300)
    plt.show()


## Load preprocessed data

These files should already exist because they were created in `02_preprocessing.ipynb`.


In [ ]:
# Salem Part files

X_train_lr_with_time = load_table("X_train_lr_with_time.csv")
X_test_lr_with_time = load_table("X_test_lr_with_time.csv")

X_train_lr_without_time = load_table("X_train_lr_without_time.csv")
X_test_lr_without_time = load_table("X_test_lr_without_time.csv")

X_train_rf_with_time = load_table("X_train_rf_with_time.csv")
X_test_rf_with_time = load_table("X_test_rf_with_time.csv")

X_train_rf_without_time = load_table("X_train_rf_without_time.csv")
X_test_rf_without_time = load_table("X_test_rf_without_time.csv")

y_train = load_y("y_train.csv")
y_test = load_y("y_test.csv")

print("X_train_lr_with_time:", X_train_lr_with_time.shape)
print("X_test_lr_with_time:", X_test_lr_with_time.shape)
print("X_train_lr_without_time:", X_train_lr_without_time.shape)
print("X_test_lr_without_time:", X_test_lr_without_time.shape)

print("X_train_rf_with_time:", X_train_rf_with_time.shape)
print("X_test_rf_with_time:", X_test_rf_with_time.shape)
print("X_train_rf_without_time:", X_train_rf_without_time.shape)
print("X_test_rf_without_time:", X_test_rf_without_time.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


In [ ]:
# Quick checks before training

assert len(X_train_lr_with_time) == len(y_train)
assert len(X_train_lr_without_time) == len(y_train)
assert len(X_train_rf_with_time) == len(y_train)
assert len(X_train_rf_without_time) == len(y_train)

assert len(X_test_lr_with_time) == len(y_test)
assert len(X_test_lr_without_time) == len(y_test)
assert len(X_test_rf_with_time) == len(y_test)
assert len(X_test_rf_without_time) == len(y_test)

assert "time" in X_train_lr_with_time.columns
assert "time" not in X_train_lr_without_time.columns
assert "time" in X_train_rf_with_time.columns
assert "time" not in X_train_rf_without_time.columns

print("All shapes and time-column checks passed.")
print("\nTraining target counts:")
print(y_train.value_counts())
print("\nTesting target counts:")
print(y_test.value_counts())


# Salem Part

## Model 1: Logistic Regression

Logistic Regression is my interpretable model.  
I use the scaled files from preprocessing because Logistic Regression works better when continuous variables are on a similar scale.


In [ ]:
# Logistic Regression with time

logistic_with_time = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

logistic_with_time.fit(X_train_lr_with_time, y_train)

logistic_with_time_results = evaluate_model(
    logistic_with_time,
    X_test_lr_with_time,
    y_test,
    model_name="Logistic Regression",
    time_version="with_time",
)

logistic_with_time_results


In [ ]:
# Classification report for Logistic Regression with time

y_pred_logistic_with_time = logistic_with_time.predict(X_test_lr_with_time)

print(classification_report(
    y_test,
    y_pred_logistic_with_time,
    target_names=["Survived", "Death"],
    zero_division=0,
))


In [ ]:
# Logistic Regression without time

logistic_without_time = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

logistic_without_time.fit(X_train_lr_without_time, y_train)

logistic_without_time_results = evaluate_model(
    logistic_without_time,
    X_test_lr_without_time,
    y_test,
    model_name="Logistic Regression",
    time_version="without_time",
)

logistic_without_time_results


In [ ]:
# Classification report for Logistic Regression without time

y_pred_logistic_without_time = logistic_without_time.predict(X_test_lr_without_time)

print(classification_report(
    y_test,
    y_pred_logistic_without_time,
    target_names=["Survived", "Death"],
    zero_division=0,
))


## Logistic Regression plots

These plots are saved into `results/figures/`.


In [ ]:
save_confusion_matrix(
    logistic_with_time,
    X_test_lr_with_time,
    y_test,
    "logistic_regression_with_time",
)

save_roc_curve(
    logistic_with_time,
    X_test_lr_with_time,
    y_test,
    "logistic_regression_with_time",
)


In [ ]:
save_confusion_matrix(
    logistic_without_time,
    X_test_lr_without_time,
    y_test,
    "logistic_regression_without_time",
)

save_roc_curve(
    logistic_without_time,
    X_test_lr_without_time,
    y_test,
    "logistic_regression_without_time",
)


## Model 2: Random Forest

Random Forest is my black-box model.  
I use the unscaled files because tree-based models do not need the continuous variables to be standardized.


In [ ]:
# Random Forest with time

random_forest_with_time = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

random_forest_with_time.fit(X_train_rf_with_time, y_train)

random_forest_with_time_results = evaluate_model(
    random_forest_with_time,
    X_test_rf_with_time,
    y_test,
    model_name="Random Forest",
    time_version="with_time",
)

random_forest_with_time_results


In [ ]:
# Classification report for Random Forest with time

y_pred_rf_with_time = random_forest_with_time.predict(X_test_rf_with_time)

print(classification_report(
    y_test,
    y_pred_rf_with_time,
    target_names=["Survived", "Death"],
    zero_division=0,
))


In [ ]:
# Random Forest without time

random_forest_without_time = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

random_forest_without_time.fit(X_train_rf_without_time, y_train)

random_forest_without_time_results = evaluate_model(
    random_forest_without_time,
    X_test_rf_without_time,
    y_test,
    model_name="Random Forest",
    time_version="without_time",
)

random_forest_without_time_results


In [ ]:
# Classification report for Random Forest without time

y_pred_rf_without_time = random_forest_without_time.predict(X_test_rf_without_time)

print(classification_report(
    y_test,
    y_pred_rf_without_time,
    target_names=["Survived", "Death"],
    zero_division=0,
))


## Random Forest plots

These plots are saved into `results/figures/`.


In [ ]:
save_confusion_matrix(
    random_forest_with_time,
    X_test_rf_with_time,
    y_test,
    "random_forest_with_time",
)

save_roc_curve(
    random_forest_with_time,
    X_test_rf_with_time,
    y_test,
    "random_forest_with_time",
)


In [ ]:
save_confusion_matrix(
    random_forest_without_time,
    X_test_rf_without_time,
    y_test,
    "random_forest_without_time",
)

save_roc_curve(
    random_forest_without_time,
    X_test_rf_without_time,
    y_test,
    "random_forest_without_time",
)


## Salem model comparison table

This table is saved as:

`results/metrics/salem_logistic_randomforest_metrics.csv`


In [ ]:
salem_results = pd.DataFrame([
    logistic_with_time_results,
    logistic_without_time_results,
    random_forest_with_time_results,
    random_forest_without_time_results,
])

salem_results = salem_results[
    ["model", "time_version", "accuracy", "precision", "recall", "f1", "roc_auc"]
]

salem_results_path = METRICS_DIR / "salem_logistic_randomforest_metrics.csv"
salem_results.to_csv(salem_results_path, index=False)

print("Saved Salem results to:", salem_results_path)
salem_results


## Short notes for Salem's part

I will use this table later in the final paper.  
For the healthcare setting, I should pay close attention to **recall**, because recall tells me how well the model finds patients who actually had a death event.


In [ ]:
# Simple automatic summary for my part

best_by_recall = salem_results.sort_values("recall", ascending=False).iloc[0]
best_by_auc = salem_results.sort_values("roc_auc", ascending=False).iloc[0]

print("Best Salem model by recall:")
print(best_by_recall)

print("\nBest Salem model by ROC-AUC:")
print(best_by_auc)


# Tala Part

## Decision Tree

Tala can use this section for Decision Tree with time and without time.


In [ ]:
# Tala Part - Decision Tree
# This section is intentionally left ready for Tala.

# Example structure:
#
# from sklearn.tree import DecisionTreeClassifier
#
# X_train_tree_with_time = load_table("X_train_tree_with_time.csv")
# X_test_tree_with_time = load_table("X_test_tree_with_time.csv")
# X_train_tree_without_time = load_table("X_train_tree_without_time.csv")
# X_test_tree_without_time = load_table("X_test_tree_without_time.csv")
#
# decision_tree_with_time = DecisionTreeClassifier(
#     max_depth=4,
#     class_weight="balanced",
#     random_state=RANDOM_STATE,
# )
# decision_tree_with_time.fit(X_train_tree_with_time, y_train)
#
# decision_tree_without_time = DecisionTreeClassifier(
#     max_depth=4,
#     class_weight="balanced",
#     random_state=RANDOM_STATE,
# )
# decision_tree_without_time.fit(X_train_tree_without_time, y_train)
#
# Then evaluate both models using evaluate_model().


## Gradient Boosting

Tala can use this section for Gradient Boosting with time and without time.


In [ ]:
# Tala Part - Gradient Boosting
# This section is intentionally left ready for Tala.

# Example structure:
#
# from sklearn.ensemble import GradientBoostingClassifier
#
# X_train_gb_with_time = load_table("X_train_gb_with_time.csv")
# X_test_gb_with_time = load_table("X_test_gb_with_time.csv")
# X_train_gb_without_time = load_table("X_train_gb_without_time.csv")
# X_test_gb_without_time = load_table("X_test_gb_without_time.csv")
#
# gradient_boosting_with_time = GradientBoostingClassifier(
#     random_state=RANDOM_STATE,
# )
# gradient_boosting_with_time.fit(X_train_gb_with_time, y_train)
#
# gradient_boosting_without_time = GradientBoostingClassifier(
#     random_state=RANDOM_STATE,
# )
# gradient_boosting_without_time.fit(X_train_gb_without_time, y_train)
#
# Then evaluate both models using evaluate_model().


# Final check

After this notebook runs, Salem's metrics should be saved in:

`results/metrics/salem_logistic_randomforest_metrics.csv`

The next notebook is:

`04_interpretability.ipynb`


In [ ]:
# Final check for Salem's output file

if salem_results_path.exists():
    print("Done. Salem model results file was created.")
    print(salem_results_path)
else:
    raise FileNotFoundError("Salem results file was not created.")
